In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Importing Data
First we import our dataset from kaggle using pandas.read_csv and visualize the header.
Our first feature is the label of the number, and the rest are the pixels that form the figure, with 0 being black and 254 being white. There are 784 pixels that form a figure of 28 x 28 px.

In [ ]:
train = pd.read_csv("../input/digit-recognizer/train.csv")
test = pd.read_csv("../input/digit-recognizer/test.csv")
train.head()

**To better understand the problem we would like to visualize one of this figures so we need a function that can plot them.**

In [ ]:
import matplotlib.pyplot as plt #Library to plot the digit

def ploting_a_digit(line,label):                       #this function takes a line from our df and plots a digit                        
    digit_array=line.to_numpy().reshape((28,28)) #Transform the pixels into a 28x28 array
    plt.title("Label: {}".format(label))          
    fig = plt.imshow(digit_array,cmap='gist_gray')     #Plot the Figure

**Now we can test our function with random numbers (every time you run the next cell you get a random number and its plot)**

In [ ]:
from random import randint

random_num=randint(0,train.shape[0]-1) # train.shape[0] - 1 as randint is inclusive in both ends
random_line=train.iloc[random_num,:]

ploting_a_digit(random_line,random_line.pop('label')) #ploting a random digit


As our first approach we will use a random forest classifier to try to predict our value. But first, we should normalize our values. For that we'll use a min max scaler on both our train and our final test set.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
labels = train.pop('label')

In [ ]:
train_set= scaler.fit_transform(train)
final_test_set=scaler.transform(test)

Now we split our train data using sklear tran_test_split so we can train our model and measure how good is working.

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(train_set,labels,test_size=0.15) #Our split will be test = 15% train = 85%

# **First Model**
Our first model, as we said already, will be a random forest classifier. This will be a starting point for our solution.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_1 = RandomForestClassifier() #Build our model
model_1.fit(X_train,y_train)       #Train our model

Now we predict using our model and then measure how good it is working using accuracy_score

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
y_pred = model_1.predict(X_test)
model_1_score = accuracy_score(y_pred,y_test)
print ("Model 1 accuracy = {:.2f}%".format(model_1_score*100))


**Our first model got a 96% of accuracy, pretty good for our first model, let's test some of our final_test data using random data.**

In [ ]:
random_num=randint(0,test.shape[0]-1) # final_test.shape[0] - 1 as randint is inclusive in both ends
random_line=test.iloc[random_num,:]

final_pred = model_1.predict(random_line.to_numpy().reshape(1,-1)) #predict on a random sample, reshape because is a single sample


ploting_a_digit(random_line,final_pred) #ploting a random prediction



# Submission to digit recognizer
We'll predict on the test data and submit our results to the Digit Recognizer competition and see how good our model works

In [ ]:
model_1_pred = model_1.predict(final_test_set) #predicting values

In [ ]:
#Fill label column on sample submission with our predictions
sample_submission = pd.read_csv("../input/digit-recognizer/sample_submission.csv",index_col='ImageId')
sample_submission.Label = model_1_pred

In [ ]:
#Output a csv to submit
sample_submission.to_csv("model_1_predict.csv")

**As we can see, most of the predicted labels are correct, let's see if we can build a model using NN that improves model_1 performance**

In [ ]:
import tensorflow as tf

model_2 = tf.keras.Sequential([
    tf.keras.layers.Dense(200,activation='selu',input_shape = (X_train.shape[-1],)),
    tf.keras.layers.Dense(100,activation = 'selu'),
    tf.keras.layers.Dense(10,activation='softmax')
])


In [ ]:
model_2.compile(loss='categorical_crossentropy',
                optimizer='Adam',
                metrics=['acc']
)

In [ ]:
model_2.summary()

In [ ]:
y_train=pd.get_dummies(y_train)
y_train.head()

In [ ]:
labels=pd.get_dummies(labels)
labels.head()

## 

In [ ]:
history_2 = model_2.fit(train_set,labels,epochs=40,validation_split=0.1)

In [ ]:
hist_pd= pd.DataFrame.from_dict(history_2.history)

In [ ]:
hist_pd.loc[:,['acc']].plot()
hist_pd.loc[:,['loss']].plot()


**With this simple NN we got a 98% accuracy, which is pretty good. Now we try some predictions**
First, We have to predict and then get the number from the softmax prediction.
After that, we fill our sample submission

In [ ]:
y_pred_2=model_2.predict(final_test_set)
salida = np.argmax(y_pred_2, axis=1)

In [ ]:
sample_submission_2 = pd.read_csv("../input/digit-recognizer/sample_submission.csv",index_col='ImageId')
sample_submission_2.Label = salida
#Output a csv to submit
sample_submission_2.to_csv("model_2_predict.csv")